# Auralis Colab Quickstart

## Standard Colab setup

Auralis supports Python 3.10, 3.11, and 3.12. Standard Colab currently runs Python 3.12, so you do not need `condacolab` or a custom Python runtime.

Run the cells in order:

1. Install the system audio dependency and pinned CUDA/PyTorch stack, then restart the runtime when prompted.
2. Install Auralis and the matching vLLM/Transformers versions.
3. Run diagnostics, upload a reference voice, and generate audio.


In [ ]:
# Step 1: install the Colab system audio dependency and PyTorch stack, then restart the runtime.
import subprocess
import sys

if not ((3, 10) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(f"Auralis supports Python >=3.10,<3.13; current Python is {sys.version}")

print(f"Python: {sys.version.split()[0]}")
print("Installing libportaudio2 for sounddevice...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "libportaudio2"], check=True)

print("Installing torch==2.5.1 and torchaudio==2.5.1 for CUDA 12.4...")

subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch", "torchaudio", "torchvision", "torchtext"
], check=False)

subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
    "torch==2.5.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)

print("PyTorch stack installed.")
print("Restart the runtime now: Runtime -> Restart session. Then run Step 2.")


In [ ]:
# Step 2: install vLLM, Transformers, and Auralis after the runtime restart.
import subprocess
import sys

AURALIS_PACKAGE = "git+https://github.com/JasperSeling/Auralis.git"
# After the upgraded package is published to PyPI, you can use: AURALIS_PACKAGE = "auralis"

subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "vllm==0.6.5",
    "transformers==4.48.2",
    AURALIS_PACKAGE,
], check=True)

# Optional but useful for English text normalization in examples.
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)

print("Auralis install finished.")


In [ ]:
# Step 3: diagnostics.
import sys

import torch
import torchaudio
import transformers
import vllm
from auralis import TTS, TTSRequest, TTSOutput

print(f"Python:      {sys.version.split()[0]}")
print(f"torch:       {torch.__version__}")
print(f"torchaudio:  {torchaudio.__version__}")
print(f"vllm:        {vllm.__version__}")
print(f"transformers:{transformers.__version__}")
print(f"CUDA:        {torch.version.cuda}")
print(f"GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")

if torch.cuda.is_available():
    print(f"GPU Memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("Import check OK.")


In [ ]:
# Step 4: upload a reference voice.
from google.colab import files
uploaded = files.upload()
ref_path = list(uploaded.keys())[0]
print(f"Reference voice: {ref_path}")


In [ ]:
# Step 5: generate speech.
from auralis import TTS, TTSRequest

tts = TTS().from_pretrained(
    "AstraMindAI/xttsv2",
    gpt_model="AstraMindAI/xtts2-gpt"
)

output = tts.generate_speech(TTSRequest(
    text="Привет! Это тест Auralis.",
    speaker_files=[ref_path],
    language="ru"
))

output.save("output.wav")
files.download("output.wav")
